# Income from crops by state, January–June 2013

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somdeepkundu/mospi-microdata-kit/blob/main/notebooks/02_nss70_farm_income_by_state.ipynb)

**Source:** NSS 70th round, Schedule 33 (Situation Assessment Survey of Agricultural Households),
Visit 2, NSO (NSSO), Ministry of Statistics and Programme Implementation.

This notebook downloads the Visit 2 Stata files (13 MB), builds one row per agricultural
household, and estimates for every state, with 95% confidence intervals:

* the share of agricultural households that grew crops in the season,
* the value of their crop output and their paid-out expenses,
* their net receipt (output minus paid-out expenses), per agricultural household and per
  cultivating household, with the median,
* the share of cultivating households whose paid-out expenses exceeded output.

Before writing anything it checks its All-India figures against NSSO's published report
KI(70/33). Only state-level aggregates leave this notebook; states with fewer than 30
cultivating sample households are suppressed.

**Time:** about 15 minutes, most of it waiting for the portal. Press **Runtime → Run all**.
Tick `use_demo_data` to try it first without an API key on synthetic data.

## 1. Install

In [ ]:
# Installs the toolkit in Colab (about 20 seconds). On your own computer, run
#   pip install "mospi-microdata-kit @ git+https://github.com/somdeepkundu/mospi-microdata-kit@v1.0.0"
# once in a terminal instead.
import importlib.util
import logging
import subprocess
import sys

PACKAGE = "mospi-microdata-kit[progress] @ git+https://github.com/somdeepkundu/mospi-microdata-kit@v1.0.0"
if importlib.util.find_spec("mospi_kit") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE], check=True)

import mospi_kit

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logging.getLogger("urllib3").setLevel(logging.WARNING)
print("mospi-microdata-kit", mospi_kit.__version__)

## 2. Choose real or demo data

In [ ]:
use_demo_data = False  #@param {type:"boolean"}

import os
use_demo_data = use_demo_data or os.environ.get("MOSPI_KIT_DEMO") == "1"
from pathlib import Path
from mospi_kit import nss70
data_dir = Path("data/nss70-visit2-demo" if use_demo_data else "data/nss70-visit2")
print("Demo data: synthetic values, never publish them." if use_demo_data else "Real data.")

## 3. Download the Stata files (skipped for demo data)

In [ ]:
if use_demo_data:
    from mospi_kit.demo import make_visit2_demo
    make_visit2_demo(data_dir)
else:
    import os
    from getpass import getpass


    def get_api_key():
        if os.environ.get("MOSPI_API_KEY"):
            return os.environ["MOSPI_API_KEY"]
        try:
            from google.colab import userdata
            return userdata.get("MOSPI_API_KEY")
        except Exception:
            return getpass("Paste your microdata.gov.in API key (hidden): ")


    client = mospi_kit.MicrodataClient(get_api_key())
    print("Ready:", client)
    client.download(nss70.VISIT2_IDNO, nss70.VISIT2_STATA_ZIP, data_dir)
    # NSSO's report, for reference
    client.download(nss70.VISIT2_IDNO, nss70.VISIT2_REPORT_PDF, data_dir)
print(sorted(p.name for p in data_dir.iterdir()))

## 4. Estimate and check against NSSO's report
`pass` means within tolerance, `warn` within twice the tolerance, `fail` further off. A run with
any `fail` is marked *unvalidated* and the project page shows it with a warning.

In [ ]:
results = nss70.run(data_dir, "results", status="demo" if use_demo_data else None)

import pandas as pd
checks = pd.DataFrame([c.__dict__ for c in results.checks])
checks[["status", "label", "expected", "observed", "tolerance", "source"]]

## 5. How the household records were put together

In [ ]:
import json
print(json.dumps(results.diagnostics, indent=1))

## 6. State estimates
Net receipt per cultivating household, ₹ for the six months, with the 95% interval. *caution*
means a relative standard error above 25%; *suppressed* means too few households to publish.

In [ ]:
t = results.table
view = t[["name", "n_cultivating", "reliability", "pct_cultivating", "net_per_cultivator",
          "net_per_cultivator_ci_low", "net_per_cultivator_ci_high", "median_net_per_cultivator",
          "unweighted_net_per_cultivator", "pct_loss"]]
view.sort_values("net_per_cultivator", ascending=False).round(0)

## 7. A quick picture to sanity-check

In [ ]:
import matplotlib.pyplot as plt

shown = t[(t.kind == "state") & (t.reliability != "suppressed")].sort_values("net_per_cultivator")
fig, ax = plt.subplots(figsize=(7, 0.28 * len(shown) + 1.2))
ax.hlines(shown.name, shown.net_per_cultivator_ci_low, shown.net_per_cultivator_ci_high,
          color="#9aa5b1", lw=2)
ax.plot(shown.net_per_cultivator, shown.name, "o", color="#1f5f4a", ms=5)
india = t.loc[t.code == "IN", "net_per_cultivator"].item()
ax.axvline(india, color="#c2410c", lw=1, ls="--")
ax.set_xlabel("Net receipt per cultivating household, Jan–Jun 2013 (₹, 95% interval)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("results/net_receipt_by_state.png", dpi=200)
plt.show()

## 8. Download the results
`results_bundle.zip` holds only state-level tables. Send `farm_income_by_state.json` to update the
project page.

In [ ]:
import zipfile
from pathlib import Path

results_dir = Path("results")
allowed = {".csv", ".json", ".png"}
leaks = [p for p in results_dir.rglob("*") if p.is_file() and p.suffix.lower() not in allowed]
assert not leaks, f"Unexpected files in results/: {leaks}"
nss70.assert_aggregate_only(results.table)

bundle = Path("results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(results_dir.iterdir()):
        zf.write(p, p.name)
print("Status:", json.loads((results_dir / "farm_income_by_state.json").read_text())["status"])
print("Bundle:", bundle.resolve())
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    pass

---
Built by Somdeep Kundu, RuDRA Lab, C-TARA, IIT Bombay. Code under the MIT licence; the survey data
belong to MoSPI and stay under its terms. Method: `docs/METHODOLOGY.md` in the repository.